In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '06_02_resting_brain_behavior_and_sensitivity.ipynb'
NOTEBOOK_RELATIVE_PATH = '06_02_resting_brain_behavior_and_sensitivity.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
import numpy as u00_np
import pandas as u00_pd
from scipy import stats as u00_stats


In [ ]:
def u00_behavior_scored_trials():
    d = u00_behavior().copy()
    d['rt_valid'] = d.correct.eq(1) & d.rt_s.notna()
    for ((sid, cond), g) in d[d.task.eq('stroop')].groupby(['subject_id', 'condition']):
        r = g.loc[g.rt_valid, 'rt_s']
        sd = r.std(ddof=1)
        if len(r) > 1:
            d.loc[g.index, 'rt_valid'] &= d.loc[g.index, 'rt_s'].between(r.mean() - 3 * sd, r.mean() + 3 * sd)
    gng = d.task.eq('gonogo')
    d.loc[gng, 'rt_valid'] &= d.loc[gng, 'rt_s'].between(0.15, 0.5)
    return d


In [ ]:
def u00_behavior_subject_scores(d=None):
    d = u00_behavior_scored_trials() if d is None else d
    rows = []
    for (sid, g) in d.groupby('subject_id'):
        st = g[g.task.eq('stroop')]
        ng = g[g.task.eq('gonogo')]
        go = ng[ng.condition.eq('go')]
        no = ng[ng.condition.eq('nogo')]
        row = {'subject_id': sid}
        for c in ['congruent', 'neutral', 'incongruent']:
            b = st[st.condition.eq(c)]
            row['stroop_' + c + '_rt_ms'] = b.loc[b.rt_valid, 'rt_s'].mean() * 1000
            row['stroop_' + c + '_accuracy'] = b.correct.mean()
        row['stroop_rt_cost_ms'] = row['stroop_incongruent_rt_ms'] - row['stroop_congruent_rt_ms']
        row['stroop_accuracy_cost_pct'] = (row['stroop_congruent_accuracy'] - row['stroop_incongruent_accuracy']) * 100
        row['stroop_mean_rt_ms'] = st.loc[st.rt_valid, 'rt_s'].mean() * 1000
        row['gonogo_go_rt_ms'] = go.loc[go.rt_valid, 'rt_s'].mean() * 1000
        row['go_accuracy'] = go.correct.mean()
        row['nogo_accuracy'] = no.correct.mean()
        row['gonogo_commission_pct'] = (1 - row['nogo_accuracy']) * 100
        row['gonogo_omission_pct'] = (1 - row['go_accuracy']) * 100
        row['gonogo_dprime'] = u00_stats.norm.ppf((go.correct.sum() + 0.5) / (len(go) + 1)) - u00_stats.norm.ppf(((1 - no.correct).sum() + 0.5) / (len(no) + 1))
        row['accuracy_difference'] = row['nogo_accuracy'] - row['go_accuracy']
        rows.append(row)
    result = u00_pd.DataFrame(rows)
    if not u00_np.isfinite(result.drop(columns='subject_id').to_numpy(float)).all():
        raise ValueError('Insufficient valid trials for a behavioral measure')
    return result


In [ ]:
from pathlib import Path as u10_Path
import sys as u10_sys
from IPython.display import display as u10_display, Image as u10_Image
import sys as u10_sys
import argparse as u10_argparse
import runpy as u10_runpy
import numpy as u10_np
import pandas as u10_pd
from scipy import stats as u10_stats


In [ ]:
def u10_brain_behavior_behavior():
    subject_scores = u00_behavior_subject_scores
    participants = u15_participants
    return participants()[['subject_id', 'age', 'sex', 'education']].merge(subject_scores(), on='subject_id', validate='one_to_one')


In [ ]:
def u10_brain_behavior_partial(x, y, covariates):
    data = u10_np.column_stack([x, y, covariates])
    data = data[u10_np.isfinite(data).all(axis=1)]
    n = len(data)
    c = u10_np.column_stack([u10_np.ones(n), data[:, 2:]])
    k = u10_np.linalg.matrix_rank(c) - 1
    if n <= k + 3:
        return dict(N=n, df=n - k - 2, r=u10_np.nan, p=u10_np.nan, CI95_low=u10_np.nan, CI95_high=u10_np.nan)
    residual = data[:, :2] - c @ u10_np.linalg.lstsq(c, data[:, :2], rcond=None)[0]
    r = u10_np.corrcoef(residual.T)[0, 1]
    df = n - k - 2
    t = r * u10_np.sqrt(df / (1 - r * r))
    ci = u10_np.tanh(u10_np.arctanh(r) + u10_np.array([-1, 1]) * u10_stats.norm.ppf(0.975) / u10_np.sqrt(n - k - 3))
    return dict(N=n, df=df, r=r, p=2 * u10_stats.t.sf(abs(t), df), CI95_low=ci[0], CI95_high=ci[1])


In [ ]:
def u10_brain_behavior_analyze(modality):
    b = u10_brain_behavior_behavior()
    outcomes = ['stroop_rt_cost_ms', 'nogo_accuracy']
    results = []
    if modality == 'task':
        motion = u10_pd.read_csv(u10_NEW / 'observed_inputs_and_model_validation.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        aggregate_task_motion = u15_aggregate_task_motion
        fd = aggregate_task_motion(motion)
        for radius in u10_ROI_VARIANTS:
            roi = u10_pd.read_csv(u10_NEW / f'independent_ALE_ROI_{radius}_mean_t.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).merge(fd, on=['subject_id', 'task'], validate='many_to_one').merge(b, on='subject_id', validate='many_to_one')
            for strict in (False, True):
                rows = []
                for ((task, key), g) in roi.groupby(['task', 'roi_id']):
                    if strict:
                        g = g[g.strict_motion]
                    cov = g[['age', 'sex', 'education', 'mean_fd_mm']].to_numpy()
                    for outcome in outcomes:
                        rows.append(dict(task=task, roi_id=key, outcome=outcome, **u10_brain_behavior_partial(g.mean_voxel_t, g[outcome], cov)))
                table = u10_pd.DataFrame(rows)
                table['p_FDR_20'] = u10_fdr(table.p)
                u10_save_table(table, u10_TASK, f'brain_behavior_5ROI_by_2task_by_2behavior_{radius}' + ('_strict_motion' if strict else ''))
    else:
        fc = u10_pd.read_csv(u10_REST_NEW / 'independent_ALE_ROI_connectivity_subject_level.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).merge(b, on='subject_id', validate='many_to_one')
        for ((radius, gsr), g) in fc.groupby(['roi_definition', 'GSR']):
            for strict in (False, True):
                group = g[g.strict_motion] if strict else g
                rows = []
                cov = group[['age', 'sex', 'education', 'mean_fd_mm']].to_numpy()
                for key in ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']:
                    for outcome in outcomes:
                        rows.append(dict(composite=key, outcome=outcome, **u10_brain_behavior_partial(group[key], group[outcome], cov)))
                table = u10_pd.DataFrame(rows)
                table['p_FDR_8'] = u10_fdr(table.p)
                name = f'brain_behavior_4_connectivity_by_2_behavior_{radius}_' + ('GSR' if gsr else 'noGSR') + ('_strict_motion' if strict else '')
                u10_save_table(table, u10_REST, name)


In [ ]:
import matplotlib.pyplot as u10_plt
from matplotlib.text import Text as u10_Text
from matplotlib.ticker import FixedLocator as u10_FixedLocator, FixedFormatter as u10_FixedFormatter


In [ ]:
def u10_apa_plain_apa(text, bold=False):
    task_pattern = 'Go/No-Go|GoNoGo|Stroop|GNG(?: [12])?'

    def fragment(part):

        def task_math(match):
            cmd = 'mathbfit' if bold else 'mathit'
            word = match[0].replace(' ', chr(92) + ' ')
            pieces = word.split('-')
            return '$' + (chr(92) + 'text{-}').join((chr(92) + cmd + '{' + piece + '}' for piece in pieces)) + '$'
        part = u10_re.sub(task_pattern, task_math, part)
        tokens = u10_re.split('(\\$[^$]*\\$)', part)
        for (i, token) in enumerate(tokens):
            if not token.startswith('$'):
                tokens[i] = u10_re.sub('\\b(r|p|t|z|Z|N|n|M|SD|SE|df|d|F|v|a)\\b', lambda m: '$' + m[0] + '$', token)
        return ''.join(tokens)
    pieces = u10_re.split('(\\$[^$]*\\$)', text)
    return ''.join((part if part.startswith('$') else fragment(part) for part in pieces))


In [ ]:
def u10_apa_apply_apa_style(fig):
    if getattr(fig, '_a8_APA_applied', False):
        return
    fig._a8_APA_applied = True
    for axis in [axis for ax in fig.axes for axis in [ax.xaxis, ax.yaxis]]:
        current = [v.get_text() for v in axis.get_ticklabels()]
        changed = [u10_apa_plain_apa(v) for v in current]
        if current != changed:
            axis.set_major_locator(u10_FixedLocator(axis.get_majorticklocs()))
            axis.set_major_formatter(u10_FixedFormatter(changed))
    for text in fig.findobj(match=u10_Text):
        value = text.get_text()
        if not value:
            continue
        out = u10_apa_plain_apa(value, bold=text.get_fontweight() in ['bold', 'heavy', 700, 800, 900])
        if out != value:
            text.set_text(out)


In [ ]:
import re as u10_re
import numpy as u10_np
import pandas as u10_pd
from matplotlib.transforms import Bbox as u10_Bbox


In [ ]:
def u10_style_apply_final_style(fig, root, name):
    if getattr(fig, '_a8_final_style_applied', False):
        return
    fig._a8_final_style_applied = True
    ax = fig.axes[0]
    if name == 'Figure_S1_representative_design_matrix_n106':
        fig._a8_tight_export = True
    elif name == 'Figure_S1_head_motion_distribution_n106':
        for c in ax.collections:
            c.set_alpha(0.9)
            c.set_edgecolors('black')
            c.set_linewidths(0.25)
    elif name.startswith('Figure_S4_'):
        top = 304 if 'Stroop' in name else 219
        fig._a8_export_bbox = u10_Bbox.from_extents(150 / 600, 38 / 600, 4191 / 600, (2100 - top) / 600)
    elif name.startswith('Figure_S2_'):
        if 'individual_Dice_distribution' in name:
            for c in ax.collections:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        elif 'activation_counts' in name:
            fig.legends[0].set_bbox_to_anchor((0.55, 0.06), transform=fig.transFigure)
    elif name.startswith('classical_brain_behavior_'):
        fig.set_size_inches(2.9, 1.9)
        rest = 'rest_connectivity' in name
        fig.subplots_adjust(left=0.37 if rest else 0.25, right=0.98, bottom=0.3, top=0.98)
        ax.xaxis.label.set_fontsize(6.5)
        ax.tick_params(labelsize=5.5)
        legend = fig.legends[0]
        legend.set_bbox_to_anchor((0.55, 0.01), transform=fig.transFigure)
        for text in legend.get_texts():
            text.set_fontsize(5.5)
        if rest:
            data = u10_pd.read_excel(u10_style_ROOT / '1.resting_state/tables/brain_behavior_4_connectivity_by_2_behavior_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            key = 'composite'
            order = ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']
        else:
            data = u10_pd.read_excel(u10_style_ROOT / '2.task_fmri/tables/brain_behavior_5ROI_by_2task_by_2behavior_ALE_3D.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            data = data[data.task.eq('stroop' if 'Stroop' in name else 'gonogo')]
            key = 'roi_id'
            order = u10_ROIS
        for (k, outcome) in enumerate(['stroop_rt_cost_ms', 'nogo_accuracy']):
            for (j, (_, row)) in enumerate(data[data.outcome.eq(outcome)].set_index(key).loc[order].iterrows()):
                xpos = float(row.CI95_high) + 0.025
                if xpos < 0 and xpos + 0.15 > 0:
                    xpos = 0.025
                ax.text(xpos, j + (k - 0.5) * 0.27, f'{float(row.r):.3f}', ha='left', va='center', fontsize=5.2, color=['#3C5488', '#E64B35'][k])
    elif name.startswith('Figure_S7_'):
        fig.set_size_inches(3.6, 2.1)
        fig.subplots_adjust(left=0.2, right=0.985, bottom=0.2, top=0.89)
        ax.set_ylabel('Cross-task pattern\nsimilarity (Fisher z)', fontsize=7, labelpad=3)
        ax.tick_params(labelsize=6)
        for c in ax.collections:
            if len(c.get_offsets()) > 100:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        data = u10_pd.read_excel(u10_style_ROOT / '2.task_fmri/tables/independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
        for (label, key) in zip(ax.texts, u10_ROIS[:3]):
            val = float(data.loc[key, 'p_positive_FDR_3'])
            label.set_text('$p_{\\mathrm{FDR}} = ' + f'{val:.3f}' + '$')
            label.set_y(1.015)
            label.set_fontsize(7.5)
    elif name in ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']:
        base = u10_style_ROOT / '2.task_fmri/tables'
        if name == 'Figure_independent_ALE_ROI_activation_n106':
            one = u10_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            for (k, task) in enumerate(u10_CONTRAST):
                for (j, (_, row)) in enumerate(one[one.task.eq(task)].set_index('roi_id').loc[u10_ROIS].iterrows()):
                    if row.p_flip_FDR_10 < 0.05:
                        ax.text(float(row.CI95_high) + 0.025, j + (k - 0.5) * 0.22, '*', fontsize=8, color=['#397A9E', '#B76B3F'][k], ha='left', va='center')
            (lo, hi) = ax.get_xlim()
            ax.set_xlim(lo, hi + 0.06)
        else:
            paired = 'paired' in name
            data = u10_pd.read_excel(base / ('independent_ALE_ROI_ALE_3D_cross_task_paired_tests.xlsx' if paired else 'independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
            for (j, key) in enumerate(u10_ROIS if paired else u10_ROIS[:3]):
                q = float(data.loc[key, 'p_FDR_5' if paired else 'p_positive_FDR_3'])
                value = '< 0.001' if q < 0.001 else '= ' + (f'{q:.5f}' if 0.05 < q < 0.0505 else f'{q:.3f}')
                ax.text(0.98, j - 0.26, '$p_{\\mathrm{FDR}} ' + value + '$', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=7.5)
    elif name in ['Figure_independent_ALE_ROI_resting_connectivity_matrix_n106', 'Figure_independent_ALE_ROI_four_prespecified_connectivity_measures_n106']:
        for text in list(fig.texts):
            text.remove()
        fig._a8_tight_export = True
        if 'matrix' in name:
            fig.set_size_inches(2.8, 2.5)
            fig.subplots_adjust(left=0.21, right=0.81, bottom=0.25, top=0.93)
            fig.axes[1].set_position([0.84, 0.25, 0.025, 0.68])
            fig.axes[1].yaxis.label.set_fontsize(6.5)
            fig.axes[1].tick_params(labelsize=5.5)
            edges = u10_pd.read_excel(u10_style_ROOT / '1.resting_state/tables/functional_connectivity_10_edges_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            lookup = {frozenset(row.edge.split('__')): row for (_, row) in edges.iterrows()}
            for text in ax.texts:
                (j, i) = map(int, text.get_position())
                row = lookup[frozenset([u10_ROIS[i], u10_ROIS[j]])]
                text.set_text(f"{float(row['mean']):.2f}" + ('*' if row.p_flip_FDR_10 < 0.05 else ''))
                text.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)
        else:
            fig.set_size_inches(2.9, 1.9)
            fig.subplots_adjust(left=0.38, right=0.98, bottom=0.23, top=0.98)
            ax.xaxis.label.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)


In [ ]:
import os as u10_os
import subprocess as u10_subprocess
import matplotlib as u10_matplotlib
u10_matplotlib.use('Agg')
import matplotlib.pyplot as u10_plt
from matplotlib.lines import Line2D as u10_Line2D
import numpy as u10_np
import pandas as u10_pd
u10_PLOT_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u10_PLOT_COLORS = {'stroop': '#397A9E', 'gonogo': '#B76B3F'}
u10_PLOT_TASK_FIGURE_NAMES = ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']
u10_PLOT_QC_FIGURE_NAMES = ['Figure_S1_representative_design_matrix_n106', 'Figure_S1_head_motion_distribution_n106']
u10_PLOT_COVERAGE_FIGURE_NAMES = ['Figure_S1_Stroop_standard_space_brain_coverage_n106', 'Figure_S1_GoNoGo_standard_space_brain_coverage_n106']
u10_PLOT_MAXT_FIGURE_NAMES = ['Figure_S4_Stroop_voxelwise_maxT_FWE_n106', 'Figure_S4_GoNoGo_voxelwise_maxT_FWE_n106']


In [ ]:
def u10_export(fig, root, name):
    u10_style_apply_final_style(fig, root, name)
    u10_apa_apply_apa_style(fig)
    out = root / 'image'
    out.mkdir(exist_ok=True)
    fig.canvas.draw()
    fig.savefig(out / (name + '.png'), dpi=600, facecolor='white', bbox_inches='tight')
    u10_plt.close(fig)


In [ ]:
import numpy as u10_np
import pandas as u10_pd
import matplotlib.pyplot as u10_plt
from matplotlib.lines import Line2D as u10_Line2D
from matplotlib.patches import Patch as u10_Patch
u10_SUPP_DICE_NAMES = ['Figure_S2_p001_individual_Dice_distribution_n106', 'Figure_S2_p001_activation_counts_n106', 'Figure_S2_p01_individual_Dice_distribution_n106', 'Figure_S2_p01_activation_counts_n106']
u10_SUPP_BEHAVIOR_NAMES = ['Supplementary_classical_brain_behavior_Stroop_activation_partial_correlations_n106', 'Supplementary_classical_brain_behavior_GoNoGo_activation_partial_correlations_n106', 'Supplementary_classical_brain_behavior_rest_connectivity_partial_correlations_n106']
u10_SUPP_PATTERN_NAME = 'Figure_S7_shared_ROI_cross_task_pattern_similarity_individual_distribution_n106'
u10_SUPP_REGION_LABELS = ['Whole brain', 'SMA', 'rAI/FO', 'lAI']
u10_SUPP_ROI_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u10_SUPP_OUTCOMES = ['stroop_rt_cost_ms', 'nogo_accuracy']
u10_SUPP_BEHAVIOR_COLORS = ['#3C5488', '#E64B35']
u10_SUPP_STATUS_COLORS = ['#3C5488', '#AAB8C3', '#E8ECEF', '#F39B7F']


In [ ]:
def u10_brain_behavior_summary(figure_names=None):
    task_path = u10_TASK / 'tables/brain_behavior_5ROI_by_2task_by_2behavior_ALE_3D.xlsx'
    rest_path = u10_REST / 'tables/brain_behavior_4_connectivity_by_2_behavior_ALE_3D_noGSR.xlsx'
    requested = set(u10_SUPP_BEHAVIOR_NAMES[2:] if figure_names is None else figure_names)
    if requested - set(u10_SUPP_BEHAVIOR_NAMES[2:]):
        raise ValueError('Unknown brain-behavior figure name')
    panels = []
    if requested.intersection(u10_SUPP_BEHAVIOR_NAMES[:2]):
        task = u10_pd.read_excel(task_path, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        assert len(task) == 20 and (not task.duplicated(['task', 'roi_id', 'outcome']).any())
        for (panel_index, task_name) in enumerate(['stroop', 'gonogo']):
            if u10_SUPP_BEHAVIOR_NAMES[panel_index] in requested:
                panels.append((panel_index, task[task.task.eq(task_name)], 'roi_id', u10_ROIS, u10_SUPP_ROI_LABELS, 'p_FDR_20'))
    if u10_SUPP_BEHAVIOR_NAMES[2] in requested:
        rest = u10_pd.read_excel(rest_path, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        assert len(rest) == 8 and (not rest.duplicated(['composite', 'outcome']).any())
        panels.append((2, rest, 'composite', ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge'], ['Shared core', 'Stroop priority', 'Go/No-Go priority', 'Bridge'], 'p_FDR_8'))
    provenance = []
    for (panel_index, data, key, order, labels, qname) in panels:
        name = u10_SUPP_BEHAVIOR_NAMES[panel_index]
        (fig, ax) = u10_plt.subplots(figsize=(3.2, 2.2))
        fig.subplots_adjust(left=0.35 if panel_index == 2 else 0.24, right=0.98, bottom=0.29, top=0.98)
        fig._a8_tight_export = True
        panel_rows = []
        for (index, outcome) in enumerate(u10_SUPP_OUTCOMES):
            sub = data[data.outcome.eq(outcome)].set_index(key).loc[order]
            values = sub.r.to_numpy()
            low = sub.CI95_low.to_numpy()
            high = sub.CI95_high.to_numpy()
            assert u10_np.isfinite(u10_np.array([values, low, high])).all()
            assert (low <= values).all() and (high >= values).all()
            yy = u10_np.arange(len(order)) + (index - 0.5) * 0.27
            color = u10_SUPP_BEHAVIOR_COLORS[index]
            ax.errorbar(values, yy, xerr=u10_np.array([values - low, high - values]), fmt='none', ecolor=color, elinewidth=0.8, capsize=2, capthick=0.7, zorder=2)
            significant = sub[qname].to_numpy() < 0.05
            for passed in [False, True]:
                keep = significant == passed
                ax.scatter(values[keep], yy[keep], s=15, facecolors=color if passed else 'white', edgecolors=color, linewidths=0.9, zorder=3)
            for (name, (_, item)) in zip(order, sub.iterrows()):
                panel_rows.append(dict(panel_task=str(item['task']) if 'task' in item.index else 'rest_noGSR', region=name, outcome=outcome, N=int(item.N), r=float(item.r), CI95_low=float(item.CI95_low), CI95_high=float(item.CI95_high), p_FDR=float(item[qname]), filled_marker=bool(item[qname] < 0.05)))
        ax.axvline(0, color='#9AA3A8', ls='--', lw=0.6, zorder=1)
        ax.set(xlim=(-0.6, 0.6), ylim=(len(order) - 0.45, -0.55), yticks=range(len(order)), yticklabels=labels, xticks=[-0.6, -0.3, 0, 0.3, 0.6])
        ax.tick_params(labelsize=6, length=2)
        ax.set_xlabel('Partial correlation r (95% CI)', fontsize=7)
        fig.legend([u10_Line2D([], [], marker='o', mfc='white', mec=c, color=c, lw=0.8, ms=3.5) for c in u10_SUPP_BEHAVIOR_COLORS], ['Stroop RT cost', 'No-Go accuracy'], loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=2, fontsize=6, handlelength=1.6, columnspacing=1.5)
        name = u10_SUPP_BEHAVIOR_NAMES[panel_index]
        u10_export(fig, u10_REST, name)
        u10_atomic_json(u10_QC / (name + '.source.json'), dict(source_sha256={str(p.name): u10_sha(p) for p in ([rest_path] if panel_index == 2 else [task_path])}, rows=panel_rows, task_tests=20, rest_tests=8, inference='Existing partial r, 95% CI and original FDR values only; no new inference'))
        provenance.extend(panel_rows)
    expected_rows = sum((8 if i == 2 else 10 for (i, n) in enumerate(u10_SUPP_BEHAVIOR_NAMES) if n in requested))
    assert len(provenance) == expected_rows


In [ ]:
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_aggregate_task_motion(motion):
    required = ['subject_id', 'task', 'mean_fd_mm', 'fd_gt_05_fraction', 'all_spikes_fraction', 'n_volumes']
    if set(required) - set(motion):
        raise ValueError('Task motion table is missing columns')
    rows = []
    for ((sid, task), g) in motion.groupby(['subject_id', 'task']):
        values = g[required[2:]].to_numpy(float)
        if not u15_np.isfinite(values).all() or g.n_volumes.le(0).any():
            raise ValueError('Invalid observed run motion statistics')
        strict = ((g.mean_fd_mm < 0.2) & (g.fd_gt_05_fraction < 0.1) & (g.all_spikes_fraction < 0.1)).all()
        rows.append(dict(subject_id=sid, task=task, mean_fd_mm=u15_np.average(g.mean_fd_mm, weights=g.n_volumes), strict_motion=bool(strict)))
    return u15_pd.DataFrame(rows)


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_behavior():
    d = u15_table('Behavior_data.csv', ['subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'], ['subject_id', 'task', 'run', 'trial_id'])
    u15_match_cohort(d, 'Behavior_data', ('subject_id', 'task', 'run', 'trial_id'))
    if not d.task.isin(['stroop', 'gonogo']).all():
        raise ValueError('task must be stroop/gonogo')
    d['rt_s'] = u15_pd.to_numeric(d.rt_s, errors='raise')
    d['response'] = u15_pd.to_numeric(d.response, errors='raise')
    d['correct'] = u15_pd.to_numeric(d.correct, errors='raise')
    if not d.correct.isin([0, 1]).all() or not d.response.dropna().isin([0, 1]).all():
        raise ValueError('correct and observed response must be binary')
    if not u15_np.isfinite(d.rt_s.dropna()).all() or (d.rt_s.dropna() <= 0).any():
        raise ValueError('RTs must be in seconds; nonresponses have NaN RT')
    for (task, conds) in [('stroop', {'congruent', 'neutral', 'incongruent'}), ('gonogo', {'go', 'nogo'})]:
        g = d[d.task.eq(task)]
        if set(g.subject_id) != set(u15_participants().subject_id) or set(g.condition) != conds:
            raise ValueError(f'{task}: incomplete subjects/conditions')
        if not g.groupby('subject_id').condition.apply(lambda x: set(x) == conds).all():
            raise ValueError(f'{task}: a participant is missing a condition')
    return d


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
from pathlib import Path as u17__A8Path
from pathlib import Path as u17_Path
import sys as u17_sys
import hashlib as u17_hashlib
import json as u17_json
import os as u17_os
for u17__key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u17_os.environ.setdefault(u17__key, '1')
import nibabel as u17_nib
import numpy as u17_np
import pandas as u17_pd
import nilearn as u17_nilearn
if u17_nilearn.__version__ != '0.13.1':
    raise RuntimeError('Use Nilearn 0.13.1, as specified in the manuscript')
from scipy import stats as u17_stats
from statsmodels.stats.multitest import multipletests as u17_multipletests
import sys as u17_sys
from pathlib import Path as u17_Path
u17_PROJECT = u15_ROOT
u17_RESULTS = u15_RESULTS
u17_DATA = u15_DATA
u17_EXPECTED_N = u15_EXPECTED_N
u17_participants = u15_participants
u17_source_signature = u15_source_signature
u17_MRI = u17_RESULTS / '3.mri_analysis'
u17_TASK = u17_MRI / '2.task_fmri'
u17_REST = u17_MRI / '1.resting_state'
u17_VERSION = 'observed_inputs_v1'
u17_LEGACY = u17_DATA / 'ale_masks'
u17_NEW = u17_TASK / 'other_outputs'
u17_REST_NEW = u17_REST / 'other_outputs'
u17_ROI_CONFIG = u17_MRI / 'ROI_definition_ALE_3D_clusters.json'
u17_ROI_VARIANTS = ('ALE_3D',)
u17_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u17_CONTRAST = {'stroop': 'incongruent_minus_congruent', 'gonogo': 'nogo_correct_minus_go_correct'}
u17_N_PERM = 50000
u17_SEED = 20261017


In [ ]:
def u17_subjects():
    return u17_participants().subject_id.tolist()


In [ ]:
def u17_sha(path):
    h = u17_hashlib.sha256()
    with u17_Path(path).open('rb') as f:
        for block in iter(lambda : f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
def u17_fingerprint(paths, extra=None):
    info = [(str(u17_Path(p).resolve()), u17_Path(p).stat().st_size, u17_Path(p).stat().st_mtime_ns) for p in paths]
    return u17_hashlib.sha256(u17_json.dumps([info, extra, u17_source_signature()], sort_keys=True).encode()).hexdigest()


In [ ]:
def u17_atomic_json(path, data):
    path = u17_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')

    def native(value):
        if isinstance(value, u17_np.generic):
            return value.item()
        if isinstance(value, u17_np.ndarray):
            return value.tolist()
        raise TypeError(f'Unsupported metadata type: {type(value).__name__}')
    tmp.write_text(u17_json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False, default=native))
    tmp.replace(path)


In [ ]:
def u17_save_table(frame, root, name):
    if '10_activation_tests' in name or name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['p_flip_positive'], errors='ignore')
    if name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    if 'pattern_similarity_' in name:
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    dest = root / 'tables'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'{name}.xlsx'
    frame.to_excel(path, index=False)
    from openpyxl import load_workbook as load_workbook
    from openpyxl.styles import Font as Font, Border as Border, Side as Side, Alignment as Alignment
    from openpyxl.utils import get_column_letter as get_column_letter
    wb = load_workbook(path)
    ws = wb.active
    rule = Side(style='thin', color='333333')
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'A2'
    for (col, key) in enumerate(frame.columns, 1):
        letter = get_column_letter(col)
        ws.column_dimensions[letter].width = min(38, max(13, len(str(key)) + 2))
        cell = ws.cell(1, col)
        cell.font = Font(name='Arial', size=10, bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical='center')
        cell.border = Border(top=rule, bottom=rule)
        for row in range(2, ws.max_row + 1):
            c = ws.cell(row, col)
            c.font = Font(name='Arial', size=10)
            if isinstance(c.value, str):
                c.alignment = Alignment(wrap_text=True, vertical='top')
                visual_length = sum((2 if ord(char) > 255 else 1 for char in c.value))
                lines = max(1, int(u17_np.ceil(visual_length / ws.column_dimensions[letter].width)))
                ws.row_dimensions[row].height = max(ws.row_dimensions[row].height or 15, min(150, lines * 15))
            if isinstance(c.value, float):
                c.number_format = '0.000E+00' if str(key).startswith(('p', 'q')) else '0.000'
            if row == ws.max_row:
                c.border = Border(bottom=rule)
    ws.row_dimensions[1].height = 32
    wb.save(path)


In [ ]:
def u17_fdr(p):
    p = u17_np.asarray(p, float)
    out = u17_np.full(p.shape, u17_np.nan)
    valid = u17_np.isfinite(p)
    if valid.any():
        out[valid] = u17_multipletests(p[valid], method='fdr_bh')[1]
    return out


In [ ]:
def u17_mean_stats(x, seed=u17_SEED, alternative='two-sided'):
    if alternative not in ('two-sided', 'greater'):
        raise ValueError('Unsupported alternative')
    key = 'p_flip' if alternative == 'two-sided' else 'p_flip_positive'
    x = u17_np.asarray(x, float)
    x = x[u17_np.isfinite(x)]
    n = len(x)
    if n < 3:
        return dict(N=n, mean=u17_np.nan, SD=u17_np.nan, t=u17_np.nan, df=n - 1, CI95_low=u17_np.nan, CI95_high=u17_np.nan, **{key: u17_np.nan})
    mu = x.mean()
    rng = u17_np.random.default_rng(seed)
    exceed = 0
    for start in range(0, u17_N_PERM, 1000):
        signs = rng.choice((-1.0, 1.0), size=(min(1000, u17_N_PERM - start), n))
        perm = signs @ x / n
        exceed += u17_np.count_nonzero(abs(perm) >= abs(mu) - 1e-14) if alternative == 'two-sided' else u17_np.count_nonzero(perm >= mu - 1e-14)
    sem = u17_stats.sem(x)
    ci = u17_stats.t.interval(0.95, n - 1, loc=mu, scale=sem)
    return dict(N=n, mean=float(mu), SD=float(x.std(ddof=1)), t=float(mu / sem), df=n - 1, CI95_low=float(ci[0]), CI95_high=float(ci[1]), **{key: (exceed + 1) / (u17_N_PERM + 1)})


In [ ]:
def u17_first_path(sid, task, stat):
    name = f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete; fallback to previous input results is prohibited: {sid}/{task}')
    return fresh


In [ ]:
def u17_new_first_complete(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    qc = u17_NEW / f'{sid}_task-{suffix}_first_level_qc.csv'
    maps = [u17_NEW / f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz' for stat in ('effect', 'variance', 'z')]
    maps.append(u17_NEW / f'{sid}_task-{task}_mask-analysis.nii.gz')
    return qc.exists() and all((p.exists() for p in maps))


In [ ]:
def u17_qc_path(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    name = f'{sid}_task-{suffix}_first_level_qc.csv'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete: {sid}/{task}')
    return fresh


In [ ]:
def u17_t_map(sid, task):
    e = u17_nib.load(u17_first_path(sid, task, 'effect'))
    v = u17_nib.load(u17_first_path(sid, task, 'variance'))
    if e.shape != v.shape or not u17_np.allclose(e.affine, v.affine):
        raise ValueError('Effect and variance image spaces do not match')
    (ed, vd) = (u17_np.asarray(e.dataobj, float), u17_np.asarray(v.dataobj, float))
    valid = u17_np.isfinite(ed) & u17_np.isfinite(vd) & (vd > 0)
    t = u17_np.divide(ed, u17_np.sqrt(u17_np.maximum(vd, 0)), out=u17_np.zeros_like(ed), where=valid)
    qc = u17_pd.read_csv(u17_qc_path(sid, task), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    dof = int((qc.n_timepoints - qc.design_rank).sum())
    return (t, valid, e, dof)


In [ ]:
def u17_roi_masks(definition='ALE_3D'):
    if definition != 'ALE_3D':
        raise ValueError('The specified analysis uses independent three-dimensional ALE cluster masks only')
    cfg = u17_json.loads(u17_ROI_CONFIG.read_text())
    manifest = u17_NEW / 'definition_and_provenance.json'
    if not manifest.exists():
        raise FileNotFoundError('Provide and validate independent ALE ROI inputs, or run the ROI definition step in this notebook')
    meta = u17_json.loads(manifest.read_text())
    if meta['config_sha256'] != u17_sha(u17_ROI_CONFIG):
        raise ValueError('ROI definition changed; regenerate ROI masks')
    ref = u17_nib.load(u17_first_path(u17_subjects()[0], 'stroop', 'effect'))
    masks = {}
    for roi in u17_ROIS:
        path = u17_NEW / f'{roi}_ALE_3D.nii.gz'
        if u17_sha(path) != meta['masks'][roi]['sha256']:
            raise ValueError('ALE mask validation failed')
        img = u17_nib.load(path)
        data = u17_np.asarray(img.dataobj)
        if img.shape != ref.shape or not u17_np.allclose(img.affine, ref.affine):
            raise ValueError('ALE ROI and subject statistic image spaces do not match')
        if not u17_np.isin(data, [0, 1]).all() or not data.any():
            raise ValueError('ALE mask must be nonempty and binary')
        masks[roi] = data > 0
    return (masks, ref)


In [ ]:
u00_behavior = u15_behavior
u10_ROOT = u15_ROOT
u10_DATA = u15_DATA
u10_RESULTS = u15_RESULTS
u10__A8Path = u17__A8Path
u10_Path = u17_Path
u10_sys = u17_sys
u10_hashlib = u17_hashlib
u10_json = u17_json
u10_os = u17_os
u10_nib = u17_nib
u10_np = u17_np
u10_pd = u17_pd
u10_nilearn = u17_nilearn
u10_stats = u17_stats
u10_multipletests = u17_multipletests
u10_PROJECT = u17_PROJECT
u10_RESULTS = u17_RESULTS
u10_DATA = u17_DATA
u10_EXPECTED_N = u17_EXPECTED_N
u10_participants = u17_participants
u10_source_signature = u17_source_signature
u10_MRI = u17_MRI
u10_TASK = u17_TASK
u10_REST = u17_REST
u10_VERSION = u17_VERSION
u10_LEGACY = u17_LEGACY
u10_NEW = u17_NEW
u10_REST_NEW = u17_REST_NEW
u10_ROI_CONFIG = u17_ROI_CONFIG
u10_ROI_VARIANTS = u17_ROI_VARIANTS
u10_ROIS = u17_ROIS
u10_CONTRAST = u17_CONTRAST
u10_N_PERM = u17_N_PERM
u10_SEED = u17_SEED
u10_subjects = u17_subjects
u10_sha = u17_sha
u10_fingerprint = u17_fingerprint
u10_atomic_json = u17_atomic_json
u10_save_table = u17_save_table
u10_fdr = u17_fdr
u10_mean_stats = u17_mean_stats
u10_first_path = u17_first_path
u10_new_first_complete = u17_new_first_complete
u10_qc_path = u17_qc_path
u10_t_map = u17_t_map
u10_roi_masks = u17_roi_masks
u10__A8Path = u17__A8Path
u10_Path = u17_Path
u10_sys = u17_sys
u10_hashlib = u17_hashlib
u10_json = u17_json
u10_os = u17_os
u10_nib = u17_nib
u10_np = u17_np
u10_pd = u17_pd
u10_nilearn = u17_nilearn
u10_stats = u17_stats
u10_multipletests = u17_multipletests
u10_PROJECT = u17_PROJECT
u10_RESULTS = u17_RESULTS
u10_DATA = u17_DATA
u10_EXPECTED_N = u17_EXPECTED_N
u10_participants = u17_participants
u10_source_signature = u17_source_signature
u10_MRI = u17_MRI
u10_TASK = u17_TASK
u10_REST = u17_REST
u10_VERSION = u17_VERSION
u10_LEGACY = u17_LEGACY
u10_NEW = u17_NEW
u10_REST_NEW = u17_REST_NEW
u10_ROI_CONFIG = u17_ROI_CONFIG
u10_ROI_VARIANTS = u17_ROI_VARIANTS
u10_ROIS = u17_ROIS
u10_CONTRAST = u17_CONTRAST
u10_N_PERM = u17_N_PERM
u10_SEED = u17_SEED
u10_subjects = u17_subjects
u10_sha = u17_sha
u10_fingerprint = u17_fingerprint
u10_atomic_json = u17_atomic_json
u10_save_table = u17_save_table
u10_fdr = u17_fdr
u10_mean_stats = u17_mean_stats
u10_first_path = u17_first_path
u10_new_first_complete = u17_new_first_complete
u10_qc_path = u17_qc_path
u10_t_map = u17_t_map
u10_roi_masks = u17_roi_masks
u10_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u10_ROIS = u17_ROIS
u10_CONTRAST = u17_CONTRAST
u10_MRI = u17_MRI
u10__A8Path = u17__A8Path
u10_Path = u17_Path
u10_sys = u17_sys
u10_hashlib = u17_hashlib
u10_json = u17_json
u10_os = u17_os
u10_nib = u17_nib
u10_np = u17_np
u10_pd = u17_pd
u10_nilearn = u17_nilearn
u10_stats = u17_stats
u10_multipletests = u17_multipletests
u10_PROJECT = u17_PROJECT
u10_RESULTS = u17_RESULTS
u10_DATA = u17_DATA
u10_EXPECTED_N = u17_EXPECTED_N
u10_participants = u17_participants
u10_source_signature = u17_source_signature
u10_MRI = u17_MRI
u10_TASK = u17_TASK
u10_REST = u17_REST
u10_VERSION = u17_VERSION
u10_LEGACY = u17_LEGACY
u10_NEW = u17_NEW
u10_REST_NEW = u17_REST_NEW
u10_ROI_CONFIG = u17_ROI_CONFIG
u10_ROI_VARIANTS = u17_ROI_VARIANTS
u10_ROIS = u17_ROIS
u10_CONTRAST = u17_CONTRAST
u10_N_PERM = u17_N_PERM
u10_SEED = u17_SEED
u10_subjects = u17_subjects
u10_sha = u17_sha
u10_fingerprint = u17_fingerprint
u10_atomic_json = u17_atomic_json
u10_save_table = u17_save_table
u10_fdr = u17_fdr
u10_mean_stats = u17_mean_stats
u10_first_path = u17_first_path
u10_new_first_complete = u17_new_first_complete
u10_qc_path = u17_qc_path
u10_t_map = u17_t_map
u10_roi_masks = u17_roi_masks
u10_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
u10_sys.dont_write_bytecode = True
u10_TASK = u17_TASK
u10_REST = u17_REST
u10_ROIS = u17_ROIS
u10_sha = u17_sha
u10_atomic_json = u17_atomic_json
u10_SUPP_REGIONS = ['whole_brain', *u10_ROIS[:3]]
u10_QC = u10_MRI / 'figure_qa'
u10_sys.dont_write_bytecode = True
u10_style_ROOT = u10_MRI
u10_sys.dont_write_bytecode = True


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u10_display = a8_display_three_line


In [ ]:
u10_brain_behavior_analyze('rest')
for u10_suffix in ['noGSR', 'noGSR_strict_motion', 'GSR', 'GSR_strict_motion']:
    u10_display(u10_pd.read_excel(u10_REST / 'tables' / f'brain_behavior_4_connectivity_by_2_behavior_ALE_3D_{u10_suffix}.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))


In [ ]:
u10_brain_behavior_summary(figure_names=['Supplementary_classical_brain_behavior_rest_connectivity_partial_correlations_n106'])
u10_display(u10_Image(filename=str(u10_REST / 'image/Supplementary_classical_brain_behavior_rest_connectivity_partial_correlations_n106.png'), width=700))
